In [63]:
import pandas as pd

In [70]:
df_time_series = (
    pd.read_parquet('../../data/gold/data_warehouse/dw_dim_time_series.parquet')
    [['time_series_id',
      'total_weeks_length',
      'num_week_with_sales',
      'avg_weekly_sales_non_zero',
      'std_weekly_sales_non_zero']])

In [71]:
df_time_series

,time_series_id,total_weeks_length,num_week_with_sales,avg_weekly_sales_non_zero,std_weekly_sales_non_zero
0,1,33,4,2.500000,2.380476
1,2,86,33,52.000000,98.889142
2,3,14,3,1.666667,0.577350
3,4,104,83,120.072289,85.469119
4,5,104,48,296.000000,967.627918
...,...,...,...,...,...
2836,2841,52,5,3.400000,4.827007
2837,2842,101,26,2.230769,1.773306
2838,2843,99,44,4.045455,4.131302
2839,2844,101,67,3.925373,2.808609


In [72]:
df_base

,time_series_id,week_date,units_sold
0,1,2023-01-09,2
1,1,2023-01-16,6
2,1,2023-01-23,0
3,1,2023-01-30,0
4,1,2023-02-06,0
...,...,...,...
203466,2845,2024-07-29,0
203467,2845,2024-08-05,0
203468,2845,2024-08-12,0
203469,2845,2024-08-19,0


In [73]:
df_base = pd.read_parquet('../../data/gold/forecasting/ds_base_dataset.parquet')[['time_series_id', 'week_date','units_sold']]
df_base

,time_series_id,week_date,units_sold
0,1,2023-01-09,2
1,1,2023-01-16,6
2,1,2023-01-23,0
3,1,2023-01-30,0
4,1,2023-02-06,0
...,...,...,...
203466,2845,2024-07-29,0
203467,2845,2024-08-05,0
203468,2845,2024-08-12,0
203469,2845,2024-08-19,0


In [74]:
df_outliers

,week_date,time_series_id,units_sold,segment,is_outlier,rolling_baseline,upper_threshold,lower_threshold,units_sold_original,units_sold_capped,units_sold_removed,units_sold_interpolated
33,2023-03-06,2,28,valid,False,20.0,60.0302,0.0,28.0,28.0,28.0,28.0
34,2023-03-13,2,11,valid,False,20.0,60.0302,0.0,11.0,11.0,11.0,11.0
35,2023-03-20,2,24,valid,False,22.0,82.0453,0.0,24.0,24.0,24.0,24.0
36,2023-03-27,2,48,valid,False,22.0,82.0453,0.0,48.0,48.0,48.0,48.0
37,2023-04-03,2,6,valid,False,22.0,82.0453,0.0,6.0,6.0,6.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
203466,2024-07-29,2845,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
203467,2024-08-05,2845,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
203468,2024-08-12,2845,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
203469,2024-08-19,2845,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [75]:
df_outliers.isna().sum()

week_date                      0
time_series_id                 0
units_sold                     0
segment                        0
is_outlier                     0
rolling_baseline           28558
upper_threshold            28558
lower_threshold            28558
units_sold_original            0
units_sold_capped              0
units_sold_removed             0
units_sold_interpolated        0
dtype: int64

In [82]:
df_base

,time_series_id,week_date,units_sold_x,units_sold_y,segment,is_outlier,rolling_baseline,upper_threshold,lower_threshold,units_sold_original,units_sold_capped,units_sold_removed,units_sold_interpolated
0,2,2023-03-06,28,28,valid,False,20.0,60.0302,0.0,28.0,28.0,28.0,28.0
1,2,2023-03-13,11,11,valid,False,20.0,60.0302,0.0,11.0,11.0,11.0,11.0
2,2,2023-03-20,24,24,valid,False,22.0,82.0453,0.0,24.0,24.0,24.0,24.0
3,2,2023-03-27,48,48,valid,False,22.0,82.0453,0.0,48.0,48.0,48.0,48.0
4,2,2023-04-03,6,6,valid,False,22.0,82.0453,0.0,6.0,6.0,6.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
186047,2845,2024-07-29,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
186048,2845,2024-08-05,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
186049,2845,2024-08-12,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
186050,2845,2024-08-19,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [76]:
df_base = pd.read_parquet('../../data/gold/forecasting/ds_base_dataset.parquet')[['time_series_id', 'week_date','units_sold']]
df_segmented = pd.read_parquet('../../data/gold/forecasting/ds_segmented_time_series.parquet')
df_base = df_base.merge(df_segmented,on='time_series_id', how='left')
df_base = df_base[df_base['segment']=='valid'].drop(columns=['segment'])
df_outliers = pd.read_parquet('../../data/gold/forecasting/ds_time_series_outliers_detected.parquet')
df_base = df_base.merge(df_outliers, on=['time_series_id', 'week_date'], how='inner').reset_index(drop=True)

In [77]:
pd.set_option('display.max_columns', None)

In [78]:
df_outliers.head()

,week_date,time_series_id,units_sold,segment,is_outlier,rolling_baseline,upper_threshold,lower_threshold,units_sold_original,units_sold_capped,units_sold_removed,units_sold_interpolated
33,2023-03-06,2,28,valid,False,20.0,60.0302,0.0,28.0,28.0,28.0,28.0
34,2023-03-13,2,11,valid,False,20.0,60.0302,0.0,11.0,11.0,11.0,11.0
35,2023-03-20,2,24,valid,False,22.0,82.0453,0.0,24.0,24.0,24.0,24.0
36,2023-03-27,2,48,valid,False,22.0,82.0453,0.0,48.0,48.0,48.0,48.0
37,2023-04-03,2,6,valid,False,22.0,82.0453,0.0,6.0,6.0,6.0,6.0


In [79]:
def classify_series_by_capped(df):
    """
    Croston demand classification (ADI × CV²).

    ADI = total periods / non-zero periods  (average demand interval)
    CV² = (std / mean)² of non-zero demand quantities
    ddof=0 avoids NaN when only one non-zero observation exists —
    a single sale has zero size variability by definition (CV²=0),
    so classification falls back to ADI only.

    Thresholds (Syntetos et al. 2005):
        ADI < 1.32  &  CV² < 0.49  → Smooth
        ADI < 1.32  &  CV² >= 0.49 → Erratic
        ADI >= 1.32 &  CV² < 0.49  → Intermittent
        ADI >= 1.32 &  CV² >= 0.49 → Lumpy
    """
    adi = df['units_sold_capped'].size / df['units_sold_capped'].astype(bool).sum()
    cv2 = (df['units_sold_capped'].std(ddof=0) / df['units_sold_capped'].mean()) ** 2 if df['units_sold_capped'].mean() != 0 else 0
    if adi < 1.32 and cv2 < 0.49:
        return 'smooth'
    elif adi < 1.32 and cv2 >= 0.49:
        return 'erratic'
    elif adi >= 1.32 and cv2 < 0.49:
        return 'intermittent'
    else:
        return 'lumpy'


In [80]:
df_base

,time_series_id,week_date,units_sold_x,units_sold_y,segment,is_outlier,rolling_baseline,upper_threshold,lower_threshold,units_sold_original,units_sold_capped,units_sold_removed,units_sold_interpolated
0,2,2023-03-06,28,28,valid,False,20.0,60.0302,0.0,28.0,28.0,28.0,28.0
1,2,2023-03-13,11,11,valid,False,20.0,60.0302,0.0,11.0,11.0,11.0,11.0
2,2,2023-03-20,24,24,valid,False,22.0,82.0453,0.0,24.0,24.0,24.0,24.0
3,2,2023-03-27,48,48,valid,False,22.0,82.0453,0.0,48.0,48.0,48.0,48.0
4,2,2023-04-03,6,6,valid,False,22.0,82.0453,0.0,6.0,6.0,6.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
186047,2845,2024-07-29,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
186048,2845,2024-08-05,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
186049,2845,2024-08-12,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
186050,2845,2024-08-19,0,0,valid,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [81]:
df_time_series['demand_type'] = df_base.groupby('time_series_id').apply(classify_series_by_capped).values
df_time_series.head()


C:\Users\eduar\AppData\Local\Temp\ipykernel_35064\4005825899.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_time_series['demand_type'] = df_base.groupby('time_series_id').apply(classify_series_by_capped).values


ValueError: Length of values (2232) does not match length of index (2841)

In [ ]:
df_time_series.pivot_table(
    index='demand_type',
    values='time_series_id',
    aggfunc='count'
)
df_time_series[df_time_series['demand_type'] == 'smooth'].describe()

In [71]:
def classify_series(df: pd.Series) -> str:
    """
    Croston demand classification (ADI × CV²).

    ADI = total periods / non-zero periods  (average demand interval)
    CV² = (std / mean)² of non-zero demand quantities
    ddof=0 avoids NaN when only one non-zero observation exists —
    a single sale has zero size variability by definition (CV²=0),
    so classification falls back to ADI only.

    Thresholds (Syntetos et al. 2005):
        ADI < 1.32  &  CV² < 0.49  → Smooth
        ADI < 1.32  &  CV² >= 0.49 → Erratic
        ADI >= 1.32 &  CV² < 0.49  → Intermittent
        ADI >= 1.32 &  CV² >= 0.49 → Lumpy
    """

    adi = df['total_weeks_length'] / df['num_week_with_sales']
    cv2 = (df['std_weekly_sales_non_zero'] / df['avg_weekly_sales_non_zero'] + 1e-6) ** 2

    if adi < 1.32:
        return 'smooth' if cv2 < 0.49 else 'erratic'
    else:
        return 'intermittent' if cv2 < 0.49 else 'lumpy'

In [72]:
df_time_series['demand_type'] = df_time_series.apply(classify_series, axis=1)

In [47]:
df_time_series

,time_series_id,total_weeks_length,num_week_with_sales,avg_weekly_sales_non_zero,std_weekly_sales_non_zero,demand_type
0,1,33,4,2.500000,2.380476,lumpy
1,2,86,33,52.000000,98.889142,lumpy
2,3,14,3,1.666667,0.577350,intermittent
3,4,104,83,120.072289,85.469119,erratic
4,5,104,48,296.000000,967.627918,lumpy
...,...,...,...,...,...,...
2836,2841,52,5,3.400000,4.827007,lumpy
2837,2842,101,26,2.230769,1.773306,lumpy
2838,2843,99,44,4.045455,4.131302,lumpy
2839,2844,101,67,3.925373,2.808609,lumpy


In [76]:
df_time_series.pivot_table(
    index='demand_type',
    values='time_series_id',
    aggfunc='count'
)

,time_series_id
demand_type,
erratic,747
intermittent,3
lumpy,1632
smooth,459


In [73]:
df_time_series.pivot_table(
    index='demand_type',
    values='time_series_id',
    aggfunc='count'
)

,time_series_id
demand_type,
erratic,849
intermittent,820
lumpy,815
smooth,357


In [56]:
df_time_series[df_time_series['demand_type'] == 'smooth'].describe()

,time_series_id,total_weeks_length,num_week_with_sales,avg_weekly_sales_non_zero,std_weekly_sales_non_zero
count,357.000000,357.000000,357.000000,357.000000,357.000000
mean,1852.131653,43.201681,40.159664,66.896626,35.724584
std,841.891672,45.393705,42.341638,259.992180,154.908422
min,9.000000,1.000000,1.000000,1.000000,0.000000
25%,1601.000000,1.000000,1.000000,2.000000,0.000000
50%,2084.000000,19.000000,17.000000,6.950000,2.851043
75%,2524.000000,104.000000,93.000000,25.600000,12.985399
max,2811.000000,104.000000,103.000000,3088.918367,1915.908453


In [48]:
df_base = df_base.merge(df_time_series[['time_series_id', 'demand_type']], on='time_series_id')
df_lumpy_base_timeseries = df_base[df_base['demand_type'] == 'lumpy']
df_intermittent_base_timeseries = df_base[df_base['demand_type'] == 'intermittent']
df_erratic_base_timeseries = df_base[df_base['demand_type'] == 'erratic']
df_smooth_base_timeseries = df_base[df_base['demand_type'] == 'smooth']

In [50]:
df_smooth_base_timeseries.head(5)

,time_series_id,week_date,supplier_id,region_id,product_id,units_sold,week,start_date,end_date,year,semester,semester_date,semester_name,quarter,quarter_date,quarter_name,month,month_name,month_date,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_weeks_ratio,sales_units,avg_weekly_sales,avg_weekly_sales_non_zero,std_weekly_sales,std_weekly_sales_non_zero,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,iqr,cv,supplier_name,region_name,product_name,product_attribute_1,product_attribute_2,product_attribute_3,demand_type
491,9,2023-05-01,1,5,185,18,18,2023-05-01,2023-05-07,2023,1,2023-05-01,H1,2,2023-04-01,Q2,5,May,2023-05-01,2023-05-01,2023-05-08,2,2,0,1.0,32,16.0,16.00,2.828427,2.828427,18,14,15,16,17,2.0,0.176777,Supplier 1,Sul,Product 185,A,A,A05,smooth
492,9,2023-05-08,1,5,185,14,19,2023-05-08,2023-05-14,2023,1,2023-05-01,H1,2,2023-04-01,Q2,5,May,2023-05-01,2023-05-01,2023-05-08,2,2,0,1.0,32,16.0,16.00,2.828427,2.828427,18,14,15,16,17,2.0,0.176777,Supplier 1,Sul,Product 185,A,A,A05,smooth
820,15,2022-10-31,1,5,289,4,44,2022-10-31,2022-11-06,2022,2,2022-10-01,H2,4,2022-10-01,Q4,10,October,2022-10-01,2022-10-31,2022-10-31,1,1,0,1.0,4,4.0,4.00,0.000000,0.000000,4,4,4,4,4,0.0,0.000000,Supplier 1,Sul,Product 289,A,A,A05,smooth
821,16,2024-07-01,1,5,294,3,27,2024-07-01,2024-07-07,2024,2,2024-07-01,H2,3,2024-07-01,Q3,7,July,2024-07-01,2024-07-01,2024-07-29,5,4,1,0.8,9,1.8,2.25,1.643168,1.500000,4,0,1,1,3,2.0,0.912870,Supplier 1,Sul,Product 294,A,A,A05,smooth
822,16,2024-07-08,1,5,294,1,28,2024-07-08,2024-07-14,2024,2,2024-07-01,H2,3,2024-07-01,Q3,7,July,2024-07-01,2024-07-01,2024-07-29,5,4,1,0.8,9,1.8,2.25,1.643168,1.500000,4,0,1,1,3,2.0,0.912870,Supplier 1,Sul,Product 294,A,A,A05,smooth


In [51]:
df_smooth_base_timeseries.to_parquet('../../data/gold/forecasting/ds_smooth_base_timeseries.parquet', index=False)
df_erratic_base_timeseries.to_parquet('../../data/gold/forecasting/ds_erratic_base_timeseries.parquet', index=False)
df_intermittent_base_timeseries.to_parquet('../../data/gold/forecasting/ds_intermittent_base_timeseries.parquet', index=False)
df_lumpy_base_timeseries.to_parquet('../../data/gold/forecasting/ds_lumpy_base_timeseries.parquet', index=False)